# 第09课 - 元认知设计模式


## 设置

本笔记本演示了使用微软代理框架的元认知设计模式。

**先决条件：**
- 通过环境变量配置的 Azure OpenAI 部署
- 通过 Azure CLI 验证身份（`az login`）


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 什么是元认知？

元认知是<strong>关于思考的思考</strong>。在人工智能代理的背景下，它意味着构建能够：

- <strong>自我反思</strong> 自己的输出和推理过程
- <strong>检测错误</strong> 并优雅地恢复，而不是默默失败
- <strong>评估</strong> 其回答是否完整且有帮助
- <strong>适应</strong> 策略，当初始方法不起作用时（例如，回退到备份系统）

一个元认知代理不仅仅回答问题——它监控自身性能并即时调整。


## 主要和备份工具

一个常见的元认知模式是<strong>回退策略</strong>。智能体首先尝试主要工具；如果失败（例如，404 错误），智能体会识别失败并透明地切换到备份工具。

这反映了现实世界的系统，其中主要服务可能不可用，智能体必须自我诊断问题然后选择替代方案。

以下我们定义了两个航班查询工具：
- <strong>主要</strong> — 覆盖巴黎、东京和巴塞罗那
- <strong>备份</strong> — 覆盖柏林、悉尼和纽约市


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## 具有错误恢复功能的自我反思代理

下面的代理被指示首先尝试主飞行系统，识别故障，并透明地切换到备用系统。在每次响应后，它会简要地自我评估是否完全回答了用户的问题。


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## 自我评估模式

元认知的另一个方面是<strong>自我评估</strong>：一个独立的代理（或同一个代理在第二遍处理时）会审查回答的完整性、准确性和有用性。

下面我们创建一个 `ResponseEvaluator` 代理，对旅行代理的回答从三个维度进行评分。


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## 总结

在本课中，您学习了如何使用 Microsoft Agent Framework 构建<strong>元认知代理</strong>：

- <strong>自我反思</strong>：监控自身推理过程并透明地传达发生内容的代理。
- <strong>带回退的错误恢复</strong>：一种主工具 + 备用工具的模式，代理检测失败（例如 404 错误）并自动尝试备用来源。
- <strong>自我评估</strong>：一个独立的评估代理，对响应的完整性、准确性和有用性进行评分。

这些模式使代理更加稳健、透明和值得信赖——这是生产部署的关键品质。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
